# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NimaWyd/Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# ── Setup — install, authenticate, connect ─────────────────────────────────
# HF_TOKEN must be a Colab Secret (plain Read token, gated-repositories
# permission). Never hardcode or print the token.
import subprocess, os
subprocess.run(["pip", "install", "duckdb", "huggingface_hub", "-q"])

import duckdb
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
except ImportError:
    _tok = os.environ.get("HF_TOKEN", "")

assert _tok, "HF_TOKEN missing — add it as a Colab Secret (Read + gated-repos) before running."
os.environ["HF_TOKEN"] = _tok   # DuckDB httpfs reads this for hf:// auth
del _tok                         # token lives in env only; not in local scope

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

BASE     = "hf://datasets/FlyRank/internship-warehouse"
FACT_MAR = f"{BASE}/fact_content_daily_performance/month=2026-03/*.parquet"
FACT_APR = f"{BASE}/fact_content_daily_performance/month=2026-04/*.parquet"

print("Connection ready.")
print("Feature window : month=2026-03")
print("Label window   : month=2026-04  (Option A — forward-looking)")

[Executed in Colab — outputs saved.]


In [ ]:
# Section 1 verification — basic sanity check on the development slice
result = con.execute(f"""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items,
        MIN(report_date)                AS min_date,
        MAX(report_date)                AS max_date
    FROM read_parquet('{FACT_MAR}')
""").df()
print("month=2026-03 overview:")
print(result.to_string(index=False))

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# Section 2 verification — list columns to confirm field classification
sample = con.execute(f"SELECT * FROM read_parquet('{FACT_MAR}') LIMIT 0").df()
print("Columns in fact_content_daily_performance:")
for col in sample.columns:
    print(f"  {col}")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Three queries prove the contract claims. Then a five-feature frame with justifications, and a deliberate leakage demonstration.*

- **Query 1 — Grain:** proves one row = one `report_date × client_hash_id × content_hash_id`
- **Query 2 — Counts + dates:** total rows, clients, content items, date span for `month=2026-03`
- **Query 3 — Availability:** rows surviving `ga4_data_available IS TRUE`
- **3b — Feature frame:** five features from March 2026, each with a "knowable because" line
- **3c — Leakage trap:** one label-derived column added, score checked, column deleted, honest score shown

In [ ]:
# ── QUERY 1 — Grain check ─────────────────────────────────────────────────
# Expect zero rows: no duplicate (client, content, date) rows in the partition.
grain_check = con.execute(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM read_parquet('{FACT_MAR}')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()

print("=== QUERY 1 — GRAIN CHECK ===")
if grain_check.empty:
    print("✓ Grain holds — zero duplicate (client, content, date) rows found.")
else:
    print(f"⚠ {len(grain_check)} duplicates — investigate.")
    print(grain_check)

# ── QUERY 2 — Row count + date span ───────────────────────────────────────
counts = con.execute(f"""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items,
        MIN(report_date)                AS min_date,
        MAX(report_date)                AS max_date
    FROM read_parquet('{FACT_MAR}')
""").df()

print("\n=== QUERY 2 — SLICE ROW COUNT + DATE SPAN ===")
print(counts.to_string(index=False))

# ── QUERY 3 — ga4_data_available IS TRUE ──────────────────────────────────
avail = con.execute(f"""
    SELECT
        COUNT(*)                                                             AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)         AS rows_ga4_true,
        ROUND(100.0 *
              SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
              / COUNT(*), 1)                                                 AS pct_ga4_available
    FROM read_parquet('{FACT_MAR}')
""").df()

print("\n=== QUERY 3 — GA4 AVAILABILITY (ga4_data_available IS TRUE) ===")
print("Rows without this flag have zero-filled GA4 columns — filter before using engagement metrics.")
print(avail.to_string(index=False))

# ── 3b. Five-feature frame ─────────────────────────────────────────────────
print("\n=== 3b. FIVE-FEATURE FRAME (feature window = month=2026-03) ===")
print("""
Feature justifications:
  avg_impressions_march  : knowable at 2026-03-31 — GSC impressions have a ~3-day lag; all March rows present.
  avg_position_march     : knowable at 2026-03-31 — position reported alongside impressions in the same GSC export.
  avg_ctr_march          : knowable at 2026-03-31 — computed from clicks ÷ impressions, both in the feature window.
  days_with_impressions  : knowable at 2026-03-31 — count of days with gsc_impressions > 0 within the same window.
  impression_consistency : knowable at 2026-03-31 — days_with_impressions ÷ days_in_month, same window.
""")

features_df = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_impressions)                                               AS avg_impressions_march,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)     AS avg_position_march,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
             ELSE NULL END                                                 AS avg_ctr_march,
        SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END)              AS days_with_impressions,
        COUNT(DISTINCT report_date)                                        AS days_in_month
    FROM read_parquet('{FACT_MAR}')
    GROUP BY client_hash_id, content_hash_id
""").df()

features_df['impression_consistency'] = (
    features_df['days_with_impressions'] / features_df['days_in_month']
)

print(f"Feature frame: {features_df.shape[0]:,} rows × {features_df.shape[1]} columns")
print(features_df[['avg_impressions_march','avg_position_march','avg_ctr_march',
                    'days_with_impressions','impression_consistency']].describe().round(3))

# Build label from April (Option A)
print("\nBuilding label from month=2026-04 (label window — never a feature)...")
label_df = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_impressions) AS avg_impressions_april
    FROM read_parquet('{FACT_APR}')
    GROUP BY client_hash_id, content_hash_id
""").df()

merged = features_df.merge(label_df, on=['client_hash_id','content_hash_id'], how='inner')
merged['is_declining'] = (
    merged['avg_impressions_april'] < merged['avg_impressions_march'] * 0.80
).astype(int)

print(f"Rows with both March + April data: {len(merged):,}")
print(merged['is_declining'].value_counts().rename({1:'declining (1)', 0:'not declining (0)'}).to_string())
print(f"Base rate: {merged['is_declining'].mean():.3f}")

# ── 3c. Leakage trap ──────────────────────────────────────────────────────
print("\n=== 3c. LEAKAGE TRAP ===")

FEATURE_COLS = ['avg_impressions_march','avg_position_march','avg_ctr_march',
                'days_with_impressions','impression_consistency']

work = merged[FEATURE_COLS + ['avg_impressions_april','is_declining']].dropna()
y    = work['is_declining'].values

# BEFORE: add avg_impressions_april (label-window data) as a feature
leaky_cols = FEATURE_COLS + ['avg_impressions_april']
X_leaky    = StandardScaler().fit_transform(work[leaky_cols].values)
lr_leaky   = LogisticRegression(max_iter=1000, random_state=42).fit(X_leaky, y)
auc_leaky  = roc_auc_score(y, lr_leaky.predict_proba(X_leaky)[:,1])
print(f"AUC WITH leaky feature (avg_impressions_april in feature set): {auc_leaky:.3f}  ← suspiciously high")

# AFTER: remove avg_impressions_april
X_clean   = StandardScaler().fit_transform(work[FEATURE_COLS].values)
lr_honest = LogisticRegression(max_iter=1000, random_state=42).fit(X_clean, y)
auc_clean = roc_auc_score(y, lr_honest.predict_proba(X_clean)[:,1])
print(f"AUC WITHOUT leaky feature (honest baseline):                  {auc_clean:.3f}")
print(f"\nDrop: Δ = {auc_leaky - auc_clean:+.3f}  — that gap is the leak, not real signal.")
print("'avg_impressions_april' removed from feature set. ✓")

## 4. Data limits

*One real, specific limitation — not a generic disclaimer.*

**Named limitation: content items with no April impressions are silently dropped from the training set.**

The forward-looking label (Option A) requires a content item to appear in both the March feature window AND the April label window. Any page that was active in March but generated zero impressions in April is excluded from `merged` at the inner-join step — not because it is stable, but because it fell out of GSC reporting entirely. This is precisely the kind of item that might be worth predicting (a page going dark is a meaningful outcome), yet it is absent from training. The code cell below quantifies the dropout. The limitation cannot be fixed by tuning the model; it would require a different label design that treats "no April data" as a positive class rather than as a dropped row.

In [ ]:
# Limitation backing: quantify silent dropout from the inner-join step
# 'features_df' has all March content items; 'merged' keeps only those with April data.
march_items  = features_df['content_hash_id'].nunique()
merged_items = merged['content_hash_id'].nunique()
dropped      = march_items - merged_items

print(f"Content items with March data:         {march_items:,}")
print(f"Content items with March + April data: {merged_items:,}")
print(f"Dropped (no April impressions):        {dropped:,}  ({100*dropped/march_items:.1f}% of March items)")
print()
print("These dropped items are not a random sample — pages that disappeared from GSC")
print("in April may be the most interesting cases to predict, yet they are absent")
print("from training. The label design, not model tuning, is where this must be fixed.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Assignment criteria:**

| Criterion | Status |
|---|---|
| Five contract answers (unit, tables, window, label/proxy, one exclusion) | ✓ Section 1 — Option A, forward-looking |
| Three queries with visible output | ✓ Section 3: grain, counts+dates, ga4 availability |
| `IS TRUE` availability check | ✓ Query 3 uses `ga4_data_available IS TRUE` |
| Five-feature frame with "knowable because" justifications | ✓ Section 3b |
| Leak-then-remove experiment with both results visible | ✓ Section 3c — AUC before and after |
| One named, specific limitation | ✓ Section 4 — silent dropout of no-April-data pages |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.